In [0]:
# openpyxl is required on Serverless compute for .xlsx parsing
%pip install openpyxl

import pandas as pd
from pyspark.sql.functions import current_date, current_timestamp, lit, regexp_extract, to_date

# Source: hospital list Excel file in the PSA landing zone
hospital_source_path = (
    "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/facilities/"
    "hospital_list_asof_20260630.xlsx"
)

# Preserve every source row as received. Positional names keep title and header rows as data.
source_sheet = pd.ExcelFile(hospital_source_path).sheet_names[0]
source_file = hospital_source_path.rsplit('/', 1)[-1]
hospital_pdf = pd.read_excel(hospital_source_path, sheet_name=source_sheet, header=None)
hospital_pdf.columns = [f'_c{i}' for i in range(hospital_pdf.shape[1])]
hospital_df = (
    spark.createDataFrame(hospital_pdf)
         .withColumn('source_file', lit(source_file))
         .withColumn('as_of_date', to_date(regexp_extract(lit(source_file), r'(?i)asof_?(\d{8})', 1), 'yyyyMMdd'))
         .withColumn('extract_date', current_date())
         .withColumn('ingested_at', current_timestamp())
)

# Write the raw source with technical ingestion metadata.
(
    hospital_df.write
               .format("delta")
               .mode("overwrite")
               .option("delta.columnMapping.mode", "name")
               .option("delta.minReaderVersion", "2")
               .option("delta.minWriterVersion", "5")
               .option("overwriteSchema", "true")
               .saveAsTable("public_health.bronze.facilities_hospital_list")
)

In [0]:
%sql
SELECT * FROM public_health.bronze.facilities_hospital_list